# Seasonal Agriculture Performance Analysis

**Student:** [Add name]  
**College:** [Add college]  
**AICTE Student ID:** [Add ID]

This notebook analyzes how farm performance changes across Kharif, Rabi and Zaid seasons using the supplied 4,000-row dataset.


## Analytical questions

1. How do yield, production, profit and resource use differ by season?
2. Are the observed yield differences statistically significant?
3. Which crops and irrigation methods perform best within the seasonal context?
4. Which measurable factors have the strongest relationship with yield and profit?
5. What actions could improve weak seasonal outcomes?


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 50)

DATA_PATH = Path("seasonal_agriculture_performance_dataset.csv")
df = pd.read_csv(DATA_PATH)
df.head()


## 1. Data understanding and quality checks

The workflow verifies dimensions, types, duplicate identifiers, missing values and core arithmetic relationships before analysis.


In [ ]:
print(f"Rows: {len(df):,}; columns: {df.shape[1]}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate Farm_ID values: {df['Farm_ID'].duplicated().sum()}")

missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("Missing values"))
display(df[["Season", "Crop", "Irrigation_Method"]].nunique().to_frame("Unique values"))


In [ ]:
# Check internal consistency with tolerances for rounded production and revenue fields.
checks = pd.Series({
    "Profit equals revenue minus cost": np.isclose(df["Profit_INR"], df["Revenue_INR"] - df["Total_Cost_INR"], atol=1).mean(),
    "Production approximates area times yield": np.isclose(df["Production_Tonnes"], df["Farm_Area_Hectares"] * df["Yield_Tonnes_Ha"], atol=0.15, equal_nan=True).mean(),
    "Revenue approximates production times price": np.isclose(df["Revenue_INR"], df["Production_Tonnes"] * df["Market_Price_INR_Tonne"], rtol=0.015, atol=100).mean(),
})
display((checks * 100).round(1).to_frame("Rows passing check (%)"))


### Cleaning decisions

- Keep one record per unique `Farm_ID`; the source contains no duplicates.
- Preserve missing values for descriptive summaries, where pandas excludes them metric by metric.
- Use median imputation only inside the predictive model so the original observations remain unchanged.
- Treat `State` and `District` as separate labels. The file contains implausible state–district combinations, so the analysis does not interpret districts as nested geography.
- Add profit margin and profit-per-hectare measures for comparable economic analysis.


In [ ]:
clean = df.copy()
clean["Profit_Margin_pct"] = np.where(clean["Revenue_INR"] != 0, 100 * clean["Profit_INR"] / clean["Revenue_INR"], np.nan)
clean["Profit_INR_Ha"] = clean["Profit_INR"] / clean["Farm_Area_Hectares"]
clean["Profitable"] = clean["Profit_INR"] > 0
clean.describe(include="all").T


## 2. Seasonal performance


In [ ]:
season_summary = clean.groupby("Season").agg(
    Farms=("Farm_ID", "count"),
    Area_ha=("Farm_Area_Hectares", "sum"),
    Avg_Yield_t_ha=("Yield_Tonnes_Ha", "mean"),
    Production_t=("Production_Tonnes", "sum"),
    Avg_Rainfall_mm=("Rainfall_mm", "mean"),
    Avg_Temperature_C=("Avg_Temperature_C", "mean"),
    Water_Used_m3=("Water_Used_m3", "sum"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Revenue_INR=("Revenue_INR", "sum"),
    Cost_INR=("Total_Cost_INR", "sum"),
    Profit_INR=("Profit_INR", "sum"),
    Profitable_Farms_pct=("Profitable", lambda x: 100*x.mean()),
    Avg_Risk_pct=("Disease_Pest_Risk_pct", "mean"),
)
season_summary["Profit_Margin_pct"] = 100 * season_summary["Profit_INR"] / season_summary["Revenue_INR"]
display(season_summary.round(2))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
order = ["Kharif", "Rabi", "Zaid"]
sns.barplot(data=clean, x="Season", y="Yield_Tonnes_Ha", order=order, estimator="mean", errorbar=("ci",95), ax=axes[0])
axes[0].set(title="Mean yield with 95% CI", ylabel="Tonnes per hectare", xlabel="")
sns.barplot(data=clean, x="Season", y="Profit_INR_Ha", order=order, estimator="mean", errorbar=("ci",95), ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(title="Mean profit per hectare", ylabel="INR per hectare", xlabel="")
sns.barplot(data=clean, x="Season", y="Water_Efficiency_t_per_1000m3", order=order, estimator="mean", errorbar=("ci",95), ax=axes[2])
axes[2].set(title="Mean water efficiency", ylabel="t per 1,000 m³", xlabel="")
plt.tight_layout()
plt.show()


## 3. Statistical comparison of yield


In [ ]:
groups = [clean.loc[clean["Season"] == s, "Yield_Tonnes_Ha"].dropna() for s in ["Kharif", "Rabi", "Zaid"]]
anova = stats.f_oneway(*groups)
kruskal = stats.kruskal(*groups)

def eta_squared(data, group, value):
    grand = data[value].mean()
    ss_between = sum(len(g) * (g[value].mean() - grand)**2 for _, g in data.dropna(subset=[value]).groupby(group))
    ss_total = ((data[value] - grand)**2).sum()
    return ss_between / ss_total

print(f"One-way ANOVA: F={anova.statistic:.3f}, p={anova.pvalue:.4g}")
print(f"Kruskal-Wallis: H={kruskal.statistic:.3f}, p={kruskal.pvalue:.4g}")
print(f"Season effect size (eta squared): {eta_squared(clean, 'Season', 'Yield_Tonnes_Ha'):.4f}")

pairs=[]
for a,b in [("Kharif","Rabi"),("Kharif","Zaid"),("Rabi","Zaid")]:
    x=clean.loc[clean.Season==a,"Yield_Tonnes_Ha"].dropna()
    y=clean.loc[clean.Season==b,"Yield_Tonnes_Ha"].dropna()
    t,p=stats.ttest_ind(x,y,equal_var=False)
    pairs.append([a,b,x.mean()-y.mean(),p,min(p*3,1)])
display(pd.DataFrame(pairs,columns=["Season A","Season B","Mean difference t/ha","Raw p","Bonferroni p"]).round(5))


The significance tests detect seasonal differences, but crop mix matters because sugarcane yields are much higher than other crops. The crop-level view below checks whether the seasonal order persists within each crop.


In [ ]:
crop_season = clean.pivot_table(index="Crop", columns="Season", values="Yield_Tonnes_Ha", aggfunc="mean").reindex(columns=order)
display(crop_season.round(2))

ax = crop_season.plot(kind="bar", figsize=(12,5))
ax.set(title="Average yield by crop and season", ylabel="Tonnes per hectare", xlabel="")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 4. Irrigation and economic performance


In [ ]:
irrigation = clean.groupby("Irrigation_Method").agg(
    Farms=("Farm_ID","count"),
    Avg_Yield_t_ha=("Yield_Tonnes_Ha","mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Profit_INR=("Profit_INR","mean"),
    Profitable_Farms_pct=("Profitable",lambda x:100*x.mean())
).sort_values("Avg_Yield_t_ha",ascending=False)
display(irrigation.round(2))

fig, axes=plt.subplots(1,2,figsize=(12,4.5))
sns.barplot(data=clean,x="Irrigation_Method",y="Yield_Tonnes_Ha",estimator="mean",errorbar=("ci",95),ax=axes[0])
axes[0].set(title="Yield by irrigation method",xlabel="",ylabel="Tonnes per hectare")
sns.barplot(data=clean,x="Irrigation_Method",y="Water_Efficiency_t_per_1000m3",estimator="mean",errorbar=("ci",95),ax=axes[1])
axes[1].set(title="Water efficiency by irrigation method",xlabel="",ylabel="t per 1,000 m³")
plt.tight_layout(); plt.show()


In [ ]:
crop_economics = clean.groupby(["Season","Crop"]).agg(
    Farms=("Farm_ID","count"),
    Avg_Yield_t_ha=("Yield_Tonnes_Ha","mean"),
    Total_Profit_INR=("Profit_INR","sum"),
    Avg_Profit_INR_Ha=("Profit_INR_Ha","mean"),
    Profitable_Farms_pct=("Profitable",lambda x:100*x.mean())
).reset_index()
display(crop_economics.sort_values(["Season","Total_Profit_INR"],ascending=[True,False]).round(2))


## 5. Factors associated with yield

A random forest estimates predictive associations after accounting for season, crop, irrigation method and numeric farm conditions. These importances support prioritization but do not establish causation.


In [ ]:
target="Yield_Tonnes_Ha"
features=["Season","Crop","Irrigation_Method","Farm_Area_Hectares","Rainfall_mm","Avg_Temperature_C","Humidity_pct","Sunlight_Hours_Day","Soil_pH","Soil_Moisture_pct","Nitrogen_kg_ha","Phosphorus_kg_ha","Potassium_kg_ha","Fertilizer_kg_ha","Pesticide_Litre_ha","Seed_Quality_Score","Water_Used_m3","Disease_Pest_Risk_pct"]
model_df=clean[features+[target]].dropna(subset=[target])
X=model_df[features]; y=model_df[target]
cat=["Season","Crop","Irrigation_Method"]
num=[c for c in features if c not in cat]
pre=ColumnTransformer([
    ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
    ("cat",OneHotEncoder(handle_unknown="ignore"),cat)
])
pipe=Pipeline([("pre",pre),("model",RandomForestRegressor(n_estimators=250,min_samples_leaf=5,random_state=42,n_jobs=-1))])
pipe.fit(X,y)
perm=permutation_importance(pipe,X,y,n_repeats=5,random_state=42,n_jobs=-1)
importance=pd.Series(perm.importances_mean,index=features).sort_values(ascending=False)
display(importance.head(12).round(4).to_frame("Permutation importance"))
importance.head(12).sort_values().plot(kind="barh",figsize=(9,5),title="Predictive importance for yield")
plt.xlabel("Decrease in model score when shuffled"); plt.tight_layout(); plt.show()


## 6. Conclusions and recommendations

### Evidence-based conclusions

- Kharif has the highest mean yield, total production, aggregate profit margin and share of profitable farms.
- Zaid has the lowest mean yield and an aggregate loss. Only about one-third of Zaid farms are profitable.
- The Kharif yield advantage appears within every crop in the crop-by-season table, so it is not explained only by a different crop mix.
- Drip irrigation has the highest mean yield and average profit. Rainfed farms show the highest water-efficiency ratio, partly because their recorded water use is lower.
- Sugarcane dominates physical yield, while chilli and sugarcane generate most aggregate profit. Maize, rice and wheat lose money on average in each season in this dataset.

### Recommendations

1. Prioritize Zaid interventions. Review crop choice, irrigation scheduling and market economics before expanding summer acreage.
2. Test drip irrigation where water access and investment costs permit, using controlled pilots within the same crop and region.
3. Track profit per hectare alongside yield. High production does not guarantee positive farm economics.
4. Investigate the repeated losses in maize, rice and wheat by separating input costs, realized prices and yield gaps.
5. Validate district-to-state mappings and collect year/date fields before using the data for geographic or time-series planning.

### Limitations

This cross-sectional dataset supports association and group comparison, not causal claims. It has no year variable, several missing observations and implausible state–district combinations. The results should guide further investigation rather than prescribe a universal farm plan.
